# Boruta vs SHAP Shootout

## What You Will Do
You will run Boruta (a wrapper method) and SHAP importance (a model-explanation method) on the same classification dataset and compare their feature selections head to head. The overlap visualisation and performance comparison will show you when they agree, when they diverge, and what each one is actually measuring.

## The Philosophical Difference
- **Boruta** asks: *does this feature carry signal beyond random noise, measured by its importance in a Random Forest?*
- **SHAP** asks: *how much does this feature move the model's prediction, averaged over all samples?*

Both answer different questions. A feature can have high SHAP importance but fail Boruta if it is only marginally better than its shadow. A feature can pass Boruta but have low SHAP if its contribution is consistent but small per sample.

## Estimated Time: under 2 minutes

---

## Setup

In [ ]:
# Purpose: Import all dependencies
# Key Concept: SHAP's TreeExplainer is exact for tree-based models, not an approximation

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import shap

from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelBinarizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

## Load the Dataset

The Wine dataset classifies 178 wine samples across 3 grape cultivars using 13 chemical measurements. It is a good test bed because some features are genuinely important, some are redundant, and the classes are moderately overlapping — so selection decisions actually matter.

In [ ]:
# Purpose: Load the Wine classification dataset

data = load_wine(as_frame=True)
X = data.data
y = data.target
feature_names = X.columns.tolist()
Xn = X.values
yn = y.values

print(f"Samples : {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes : {dict(zip(data.target_names, np.bincount(yn)))}")
print(f"Features: {feature_names}")

## Method 1 — Boruta

We run 40 trials. A feature passes if it beats the shadow maximum in more than 60% of trials — a tighter threshold than the default 50%, which gives us a confident-positive selection set.

In [ ]:
# Purpose: Run Boruta shadow-feature selection
# Key Concept: Shadow features are column-permuted copies — they carry zero information.
#              Consistently beating them across 40 trials is a strong significance test.

def boruta_select(X_arr, y_arr, n_trials=40, win_threshold=0.6, random_state=42):
    """
    Select features using shadow-feature comparison over multiple RF trials.

    Returns
    -------
    selected_mask : bool array shape (n_features,)
    wins : int array shape (n_features,) — number of trials each feature beat shadow max
    win_fractions : float array shape (n_features,)
    """
    rng = np.random.RandomState(random_state)
    n_features = X_arr.shape[1]
    wins = np.zeros(n_features, dtype=int)

    rf = RandomForestClassifier(
        n_estimators=50, random_state=random_state, n_jobs=-1
    )

    for _ in range(n_trials):
        X_shadow = np.column_stack([
            rng.permutation(X_arr[:, col]) for col in range(n_features)
        ])
        X_aug = np.hstack([X_arr, X_shadow])
        rf.set_params(random_state=rng.randint(0, 100000))
        rf.fit(X_aug, y_arr)

        real_imp = rf.feature_importances_[:n_features]
        shadow_max = rf.feature_importances_[n_features:].max()
        wins += (real_imp > shadow_max).astype(int)

    win_fractions = wins / n_trials
    selected_mask = win_fractions >= win_threshold
    return selected_mask, wins, win_fractions


boruta_mask, boruta_wins, boruta_fractions = boruta_select(
    Xn, yn, n_trials=40, win_threshold=0.6
)
boruta_selected = [feature_names[i] for i in np.where(boruta_mask)[0]]

print(f"Boruta selected {len(boruta_selected)} / {len(feature_names)} features:")
for name in boruta_selected:
    frac = boruta_fractions[feature_names.index(name)]
    print(f"  {name:<30}  win fraction = {frac:.2f}")

## Method 2 — SHAP Importance

We train one Random Forest on the full dataset and use `shap.TreeExplainer` to compute exact Shapley values. The mean absolute SHAP value per feature measures average contribution to model predictions. We select features whose mean |SHAP| exceeds the median — the top half by contribution.

In [ ]:
# Purpose: Compute SHAP importance values and derive a feature selection threshold
# Key Concept: SHAP values are additive — they decompose each prediction into
#              per-feature contributions that sum to the model output

# Train RF on all features
rf_full = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf_full.fit(Xn, yn)

# Compute SHAP values using TreeExplainer (exact, no approximation for RF)
explainer = shap.TreeExplainer(rf_full)
shap_values = explainer.shap_values(Xn)  # shape: (n_samples, n_features, n_classes)

# Average across classes: mean of per-class |SHAP|
# shap_values shape: (n_samples, n_features, n_classes)
shap_arr = np.array(shap_values)
if shap_arr.ndim == 3:
    mean_abs_shap = np.abs(shap_arr).mean(axis=0).mean(axis=1)  # (n_features,)
else:
    mean_abs_shap = np.abs(shap_arr).mean(axis=0)

# Select top half by SHAP importance
shap_threshold = np.median(mean_abs_shap)
shap_mask = mean_abs_shap >= shap_threshold
shap_selected = [feature_names[i] for i in np.where(shap_mask)[0]]

print(f"SHAP selected {len(shap_selected)} / {len(feature_names)} features "
      f"(threshold = {shap_threshold:.4f}):")
shap_ranking = np.argsort(mean_abs_shap)[::-1]
for rank, idx in enumerate(shap_ranking, start=1):
    marker = '*' if feature_names[idx] in shap_selected else ' '
    print(f"  {marker} {rank:2d}. {feature_names[idx]:<30}  mean|SHAP| = {mean_abs_shap[idx]:.4f}")

## Head-to-Head Comparison: Overlap Visualisation

We visualise the overlap with a Venn-style diagram and a feature-by-feature comparison table. Features in both sets are the high-confidence selections.

In [ ]:
# Purpose: Show which features each method selects and where they agree
# Key Concept: Disagreement between methods is informative — it points to features
#              that are borderline, context-dependent, or measuring different things

boruta_set = set(boruta_selected)
shap_set = set(shap_selected)
both_set = boruta_set & shap_set
boruta_only = boruta_set - shap_set
shap_only = shap_set - boruta_set
neither_set = set(feature_names) - boruta_set - shap_set

print("FEATURE SELECTION COMPARISON")
print('=' * 55)
print(f"  Both Boruta AND SHAP ({len(both_set)}):")
for f in sorted(both_set):
    print(f"    + {f}")
print(f"\n  Boruta ONLY ({len(boruta_only)}):")
for f in sorted(boruta_only):
    frac = boruta_fractions[feature_names.index(f)]
    print(f"    B {f}  (Boruta win={frac:.2f})")
print(f"\n  SHAP ONLY ({len(shap_only)}):")
for f in sorted(shap_only):
    shap_v = mean_abs_shap[feature_names.index(f)]
    print(f"    S {f}  (SHAP={shap_v:.4f})")
print(f"\n  Neither ({len(neither_set)}):")
for f in sorted(neither_set):
    print(f"    - {f}")

In [ ]:
# Purpose: Visual Venn diagram and per-feature comparison bars
# Key Concept: Visual inspection reveals patterns that tables hide

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Venn-style overlap diagram using circles
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')

circle_b = Circle((3.5, 5), 3, fill=True, alpha=0.35, color='#d62728', label=f'Boruta ({len(boruta_selected)})')
circle_s = Circle((6.5, 5), 3, fill=True, alpha=0.35, color='#1f77b4', label=f'SHAP ({len(shap_selected)})')
ax.add_patch(circle_b)
ax.add_patch(circle_s)

# Text labels
ax.text(2.2, 5, f'Boruta\nonly\n({len(boruta_only)})', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#d62728')
ax.text(5, 5, f'Both\n({len(both_set)})', ha='center', va='center',
        fontsize=11, fontweight='bold', color='black')
ax.text(7.8, 5, f'SHAP\nonly\n({len(shap_only)})', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1f77b4')

ax.set_title(f'Feature Selection Overlap\n(total features = {len(feature_names)})', fontsize=12)
ax.legend(loc='upper right', fontsize=10)
ax.axis('off')

# Right: Per-feature comparison — Boruta win fraction vs normalised SHAP
ax2 = axes[1]

norm_shap = mean_abs_shap / mean_abs_shap.max()

# Sort by Boruta win fraction
sort_idx = np.argsort(boruta_fractions)[::-1]
sorted_names = [feature_names[i] for i in sort_idx]
sorted_boruta = boruta_fractions[sort_idx]
sorted_shap = norm_shap[sort_idx]

x = np.arange(len(feature_names))
width = 0.38
ax2.bar(x - width/2, sorted_boruta, width, label='Boruta win fraction', color='#d62728', alpha=0.8)
ax2.bar(x + width/2, sorted_shap, width, label='Normalised SHAP', color='#1f77b4', alpha=0.8)
ax2.axhline(y=0.6, color='#d62728', linestyle='--', alpha=0.6, linewidth=1.2)
ax2.axhline(y=np.median(norm_shap), color='#1f77b4', linestyle='--', alpha=0.6, linewidth=1.2)
ax2.set_xticks(x)
ax2.set_xticklabels(sorted_names, rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('Score (Boruta: win fraction | SHAP: normalised)', fontsize=9)
ax2.set_title('Boruta vs SHAP Score per Feature\n(dashed = selection threshold)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Performance Comparison

Now we validate whether the selection differences translate into accuracy differences. We test four feature sets: all features, Boruta-selected, SHAP-selected, and the consensus (both methods agree).

In [ ]:
# Purpose: Compare cross-validated accuracy across the four feature subsets
# Key Concept: Performance comparison is the ground truth; agreement between
#              methods only matters if it predicts better generalisation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_eval = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)

def get_indices(name_list):
    return [feature_names.index(n) for n in name_list]

subsets = {
    f'All {len(feature_names)} features': get_indices(feature_names),
    f'Boruta ({len(boruta_selected)})': get_indices(boruta_selected),
    f'SHAP top-half ({len(shap_selected)})': get_indices(shap_selected),
    f'Consensus — both ({len(both_set)})': get_indices(list(both_set)),
}

perf = {}
for label, idx in subsets.items():
    if len(idx) == 0:
        perf[label] = (0.0, 0.0)
        continue
    sc = cross_val_score(rf_eval, Xn[:, idx], yn, cv=cv, scoring='accuracy')
    perf[label] = (sc.mean(), sc.std())

print(f"{'Feature Set':<35} {'Acc Mean':>9} {'Std':>7} {'n_feat':>8}")
print('-' * 62)
for label, (mean, std) in perf.items():
    idx = subsets[label]
    print(f"{label:<35} {mean:>9.4f} {std:>7.4f} {len(idx):>8}")

# SHAP beeswarm for consensus features
consensus_idx = get_indices(list(both_set))
if len(consensus_idx) > 0:
    rf_consensus = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
    rf_consensus.fit(Xn[:, consensus_idx], yn)
    explainer_c = shap.TreeExplainer(rf_consensus)
    shap_consensus = explainer_c.shap_values(Xn[:, consensus_idx])
    shap_consensus_arr = np.array(shap_consensus)
    print(f"\nConsensus model trained on {len(consensus_idx)} features.")

## SHAP Summary Plot for Consensus Features

A beeswarm plot shows not just which features matter but **how** they matter: positive SHAP values push predictions toward a class, negative values push away. This is information Boruta cannot provide — it only answers yes/no, not direction or magnitude per sample.

In [ ]:
# Purpose: SHAP beeswarm plot for the consensus feature set
# Key Concept: SHAP beeswarms show feature effect direction and sample-level distribution,
#              which is actionable insight that feature importance scores hide

if len(consensus_idx) > 0:
    consensus_feature_names = [feature_names[i] for i in consensus_idx]
    shap_c_arr = np.array(shap_consensus)

    if shap_c_arr.ndim == 3:
        # Multi-class: show SHAP for class 0
        shap_for_plot = shap_c_arr[:, :, 0]
        class_label = data.target_names[0]
    else:
        shap_for_plot = shap_c_arr
        class_label = 'positive class'

    mean_abs_c = np.abs(shap_for_plot).mean(axis=0)
    sort_order = np.argsort(mean_abs_c)[::-1]

    fig, ax = plt.subplots(figsize=(10, max(4, len(consensus_idx) * 0.5 + 1)))

    for row_pos, feat_idx in enumerate(sort_order):
        shap_col = shap_for_plot[:, feat_idx]
        feat_vals = Xn[:, consensus_idx[feat_idx]]
        feat_norm = (feat_vals - feat_vals.min()) / (feat_vals.ptp() + 1e-9)
        scatter = ax.scatter(
            shap_col,
            np.full_like(shap_col, row_pos) + np.random.uniform(-0.15, 0.15, len(shap_col)),
            c=feat_norm, cmap='coolwarm', alpha=0.5, s=12, linewidths=0
        )

    ax.set_yticks(range(len(sort_order)))
    ax.set_yticklabels([consensus_feature_names[i] for i in sort_order], fontsize=10)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel(f'SHAP value (impact on prediction of class: {class_label})', fontsize=10)
    ax.set_title('SHAP Beeswarm — Consensus Features\n'
                 '(colour = feature value: red=high, blue=low)', fontsize=11)
    plt.colorbar(scatter, ax=ax, label='Feature value (normalised)')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Exercise: Shift the Boruta Threshold and Observe the Effect

**Task:** Rerun Boruta with `win_threshold=0.4` (more permissive) and `win_threshold=0.8` (more strict). For each threshold, compute the overlap with SHAP and the 5-fold accuracy of the resulting subset.

**Expected outcome:** Lower threshold = more features selected, higher overlap with SHAP, but smaller accuracy gain over baseline. Higher threshold = fewer features, potentially less overlap, possibly higher precision.

<details>
<summary>Hint 1</summary>
You can reuse boruta_wins from the already-computed 40-trial run. Just change the threshold: boruta_mask_new = boruta_fractions >= new_threshold
</details>

<details>
<summary>Hint 2 (more specific)</summary>
To compute overlap: set(boruta_selected_new) & shap_set — intersection of both sets.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

# Thresholds to test — modify this list
thresholds = [0.4, 0.5, 0.6, 0.7, 0.8]

threshold_results = []
for thresh in thresholds:
    mask = boruta_fractions >= thresh
    selected = [feature_names[i] for i in np.where(mask)[0]]

    overlap = set(selected) & shap_set

    if len(selected) > 0:
        sc = cross_val_score(
            rf_eval, Xn[:, [feature_names.index(f) for f in selected]],
            yn, cv=cv, scoring='accuracy'
        )
        acc = sc.mean()
    else:
        acc = 0.0

    threshold_results.append({
        'threshold': thresh,
        'n_selected': len(selected),
        'shap_overlap': len(overlap),
        'accuracy': acc,
    })
    print(f"thresh={thresh:.1f}: {len(selected):2d} features, overlap={len(overlap)}, acc={acc:.4f}")

# Which threshold gives the best accuracy?
best_row = max(threshold_results, key=lambda r: r['accuracy'])
your_best_threshold = best_row['threshold']
print(f"\nBest threshold: {your_best_threshold} with accuracy {best_row['accuracy']:.4f}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise():
    assert len(threshold_results) == len(thresholds), \
        "Run the threshold loop for all values in the thresholds list."
    assert your_best_threshold is not None, "Set your_best_threshold."
    assert your_best_threshold in thresholds, \
        "your_best_threshold should be one of the tested values."

    accs = [r['accuracy'] for r in threshold_results]
    assert max(accs) > 0.93, \
        f"Best accuracy {max(accs):.4f} seems low — check the cross-validation logic."

    # Verify monotonic relationship: higher threshold -> fewer features
    counts = [r['n_selected'] for r in threshold_results]
    is_weakly_decreasing = all(counts[i] >= counts[i+1] for i in range(len(counts)-1))
    assert is_weakly_decreasing, \
        "Higher thresholds should select fewer or equal features — check the mask computation."

    print(f"Exercise passed! Best threshold = {your_best_threshold}, "
          f"accuracy = {best_row['accuracy']:.4f}.")

test_exercise()

## Summary

**Key Takeaways:**

1. **Boruta** is a statistical test: a feature passes if it consistently outperforms random noise (its shadow copy). It answers yes/no.
2. **SHAP** is a contribution measure: it quantifies how much each feature moves model predictions, per sample, with direction. It answers how much and which way.
3. **Consensus features** (selected by both) are the highest-confidence candidates.
4. **Disagreements reveal insight**: Boruta-only features may have weak but consistent importance; SHAP-only features may have high average impact but fail the strict shadow-beat test.
5. **Neither method is universally better** — the right choice depends on whether you want a significance test or an explanation.

**What is next:**
- `ga_feature_selector_quickstart.ipynb` — evolutionary search for the optimal subset
- Module 4 (Embedded Methods) for SHAP-based selection theory
- Module 3 (Wrapper Methods) for the statistical foundations of Boruta